# 05 — Learning curve (Giai đoạn 5)

## Chuẩn bị môi trường và chạy thử

Trước khi chạy learning curve chính thức, em muốn chắc rằng môi trường trên Kaggle thực sự chạy được từ đầu đến cuối: lấy được mã
nguồn của đề tài, đọc đúng dữ liệu đã gán nhãn, có GPU, tải được ViSoBERT và huấn luyện được ít nhất một lượt nhỏ. Learning curve
cần huấn luyện nhiều lần (nhiều kích thước tập Train, mỗi kích thước ba seed), nên nếu đến lúc đó mới phát hiện thiếu thư viện hay
sai đường dẫn thì sẽ mất rất nhiều thời gian GPU. Vì vậy ở nhịp này em chỉ làm một lần chạy thử rất nhỏ, kết quả của nó không được
dùng cho bất kỳ phân tích nào.

Notebook được bật GPU và Internet, vì em cần clone mã nguồn từ GitHub và tải trọng số ViSoBERT từ Hugging Face.

Mã nguồn của đề tài nằm trong một repository riêng tư trên GitHub, nên em lưu token truy cập trong mục Secrets của Kaggle thay vì
ghi thẳng vào notebook. Sau khi clone, em đặt lại địa chỉ remote về dạng không chứa token, vì Kaggle lưu lại thư mục
`/kaggle/working` khi lưu phiên bản notebook và em không muốn token nằm trong đó. Lưu ý là bản trên GitHub chỉ có những gì em đã
push; ở bước này em chỉ cần các tiện ích chung trong `src.utils`.

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    # clone im lặng để token không xuất hiện trong log
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
# gỡ token khỏi .git/config (thư mục /kaggle/working được lưu cùng phiên bản notebook)
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))

from src.utils.seeds import set_seed
from src.utils.kaggle_paths import is_kaggle
import yaml
set_seed(42)
commit = subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip()
print('Import src.utils: OK | Kaggle:', is_kaggle(), '| commit:', commit)
CFG = yaml.safe_load(open(os.path.join(REPO_DIR, '03_configs', 'frozen', 'classification.yaml'), encoding='utf-8'))
print('Cấu hình ViSoBERT:', CFG['strategy_b_visobert'])

## Dữ liệu

Trên Kaggle em chỉ đưa lên hai tập dùng cho phát triển mô hình là Train và Validation. Hold-out, focused set và Lazada không có mặt
ở đây, đúng với thiết kế: các tập này bị khóa cho đến lần đánh giá cuối cùng nên không được xuất hiện trong môi trường huấn luyện.
Em kiểm tra lại kích thước và phân bố nhãn để chắc rằng đây đúng là phiên bản sau khi cân lại Validation (amendment v1.8):
Train 1.200 review, Validation 350 review với 22 review FEATURE.

In [ ]:
import pandas as pd
def find_data_dir(root='/kaggle/input', target='shopee_train.csv'):
    """Tìm thư mục chứa shopee_train.csv trong /kaggle/input (slug dataset có thể khác nhau)."""
    for dirpath, _, files in os.walk(root):
        if target in files:
            return dirpath
    found = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
    raise FileNotFoundError(f'Không tìm thấy {target} trong {root}. Các file hiện có:\n' +
                            ('\n'.join(found) if found else '(trống — chưa gắn dataset vào notebook)'))

DATA_DIR = find_data_dir()
print('DATA_DIR:', DATA_DIR)
train_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA_DIR, 'shopee_validation.csv'), encoding='utf-8-sig')
order = ['BUG', 'FEATURE', 'UX', 'OTHER']
print(pd.DataFrame({'train': train_df['gold_label'].value_counts().reindex(order),
                    'validation': val_df['gold_label'].value_counts().reindex(order)}).to_string())
print('Train:', len(train_df), '| Validation:', len(val_df), '| FEATURE trong Validation:', int((val_df['gold_label'] == 'FEATURE').sum()))
assert len(train_df) == 1200 and len(val_df) == 350 and (val_df['gold_label'] == 'FEATURE').sum() == 22
assert not set(train_df['review_id']) & set(val_df['review_id']), 'Train và Validation trùng review'
print('Các file trong dataset:', sorted(os.listdir(DATA_DIR)))

## GPU và thư viện

Tiếp theo em kiểm tra GPU và tokenizer của ViSoBERT (`uitnlp/visobert`). Repo của ViSoBERT chỉ có file `sentencepiece.bpe.model`
(SentencePiece kiểu BPE). Ở lần chạy đầu em dùng `AutoTokenizer`, và ở Giai đoạn 6 em phát hiện `transformers` 5.x dựng lại file này
thành tokenizer kiểu Unigram nên tách từ sai: "thủ tục" thành "▁thủ ▁t ục", chỉ khoảng 8% review Train được tách giống bản gốc. Vì vậy
em mã hóa trực tiếp bằng `sentencepiece` (`src.classification.strategy_visobert.tokenizer`), cho cả lúc train lẫn lúc dự đoán.

Cell dưới kiểm tra ba điều. Một là cách tách của em khớp với `spm.encode` gốc trên vài câu mẫu có teencode và emoji. Hai là số token
trung bình mỗi review Train, khoảng 20–21 với tokenizer đúng, so với khoảng 33 khi tách sai. Ba là id em đưa vào có trỏ đúng hàng
embedding mà mô hình đã học lúc pretrain hay không: em che lần lượt từng token và để đầu MLM của ViSoBERT đoán lại. Nếu id đúng, mô
hình đoán lại được khoảng 40% token; nếu lệch offset, tỉ lệ này rơi về khoảng 1%.

In [ ]:
import torch, transformers, sentencepiece, numpy as np
from src.classification.strategy_visobert.tokenizer import VisobertTokenizer, self_check, embedding_alignment
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có GPU')
assert torch.cuda.is_available(), 'Chưa bật GPU cho notebook'
MODEL_NAME = CFG['strategy_b_visobert']['model_name']
tokenizer = VisobertTokenizer(MODEL_NAME)
self_check(tokenizer)
sample = 'app cứ bị đứng khi thanh toán, sp thì ok nhưng giao hơi lâu 😡'
print('Câu mẫu:', sample)
print('Token:', tokenizer.convert_ids_to_tokens(tokenizer([sample])['input_ids'][0].tolist()))
print('Token trung bình / review Train:', round(np.mean([len(tokenizer.encode(t)) for t in train_df['text'].astype(str)]), 1))
align = embedding_alignment(tokenizer, MODEL_NAME, train_df['text'].astype(str).sample(60, random_state=42))
print(f"MLM đoán lại token bị che: {align['mlm_top1']:.1%} trên {align['n_tokens']} token")
assert align['mlm_top1'] > 0.2, 'id token không khớp embedding của ViSoBERT'

## Chạy thử một lượt huấn luyện

Để kiểm tra toàn bộ đường đi từ dữ liệu đến dự đoán, em huấn luyện ViSoBERT cho tầng 1 (ACTIONABLE / NOT_ACTIONABLE), suy ra từ nhãn
4 lớp đúng như protocol: BUG, FEATURE và UX là ACTIONABLE, OTHER là NOT_ACTIONABLE. Em chỉ dùng 100 review đầu tiên của Train sau khi
xáo trộn với seed 42, huấn luyện đúng một epoch, rồi dự đoán trên 20 review Validation. Với lượng dữ liệu này mô hình chưa học được
gì đáng kể, và em cũng không dùng kết quả này để đánh giá; mục đích duy nhất là xác nhận việc tải mô hình, huấn luyện và dự đoán chạy
không lỗi, đồng thời đo thời gian để ước lượng chi phí GPU cho learning curve.

Vòng huấn luyện ở đây em viết thẳng trong cell vì nó chỉ dùng cho lần chạy thử; phần huấn luyện chính thức của các chiến lược sẽ nằm
trong `src/` như các bước trước.

In [ ]:
import time, numpy as np
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

ACTIONABLE = {'BUG', 'FEATURE', 'UX'}
to_tier1 = lambda lab: int(lab in ACTIONABLE)            # 1 = ACTIONABLE, 0 = NOT_ACTIONABLE
MAX_LEN, LR, BATCH = CFG['strategy_b_visobert']['max_len'], CFG['strategy_b_visobert']['lr'], 16
device = torch.device('cuda')

set_seed(42)
smoke_train = train_df.sample(n=100, random_state=42)
smoke_val = val_df.sample(n=20, random_state=42)

def batches(df, shuffle):
    data = list(zip(df['text'].astype(str), df['gold_label'].map(to_tier1)))
    def collate(b):
        enc = tokenizer([t for t, _ in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        enc['labels'] = torch.tensor([y for _, y in b])
        return enc
    return DataLoader(data, batch_size=BATCH, shuffle=shuffle, collate_fn=collate,
                      generator=torch.Generator().manual_seed(42))

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

torch.cuda.synchronize(); t0 = time.time()
model.train()
for b in batches(smoke_train, shuffle=True):
    b = {k: v.to(device) for k, v in b.items()}
    loss = model(**b).loss
    loss.backward(); opt.step(); opt.zero_grad()
torch.cuda.synchronize(); train_seconds = time.time() - t0
print(f'1 epoch trên {len(smoke_train)} review: {train_seconds:.1f} giây | loss cuối: {loss.item():.4f}')

model.eval(); preds = []
t0 = time.time()
with torch.no_grad():
    for b in batches(smoke_val, shuffle=False):
        b = {k: v.to(device) for k, v in b.items() if k != 'labels'}
        preds += model(**b).logits.argmax(-1).tolist()
infer_seconds = time.time() - t0
names = {1: 'ACTIONABLE', 0: 'NOT_ACTIONABLE'}
out = smoke_val.assign(tier1_gold=smoke_val['gold_label'].map(lambda l: names[to_tier1(l)]),
                       tier1_pred=[names[p] for p in preds])
print(f'Dự đoán {len(smoke_val)} review Validation: {infer_seconds:.2f} giây')
print(out[['text', 'tier1_gold', 'tier1_pred']].head(8).assign(text=lambda d: d['text'].str[:70]).to_string(index=False))
print('Peak GPU memory:', round(torch.cuda.max_memory_allocated() / 2**30, 2), 'GB')

## Nhận xét về môi trường

Cell cuối cùng tổng hợp lại các lần kiểm tra ở trên và dùng thời gian đo được để ước lượng thời gian GPU cho learning curve. Ước
lượng này dựa trên giả định thời gian huấn luyện tăng gần tuyến tính theo số review, với các mốc 400, 800 và 1.200 review (Train
hiện có 1.200), mỗi mốc ba seed và bốn epoch theo cấu hình, cho cả hai tầng. Đây là cận trên, vì tầng 2 chỉ huấn luyện trên phần
actionable, ít hơn nhiều so với toàn bộ Train. Em sẽ ghi lại nhận xét cụ thể sau khi xem kết quả chạy trên Kaggle.

In [ ]:
per_review_epoch = train_seconds / len(smoke_train)
sizes, seeds, epochs, tiers = [400, 800, 1200], 3, CFG['strategy_b_visobert']['epochs'], 2
est = per_review_epoch * sum(sizes) * seeds * epochs * tiers
print(f'Môi trường: clone mã nguồn OK, dữ liệu OK (Train {len(train_df)}, Validation {len(val_df)}), '
      f'GPU {torch.cuda.get_device_name(0)}, tokenizer và mô hình ViSoBERT tải được, train + predict chạy không lỗi.')
print(f'Thời gian 1 epoch / 100 review: {train_seconds:.1f} giây (~{per_review_epoch * 1000:.0f} giây cho 1.000 review mỗi epoch).')
print(f'Ước lượng cận trên cho learning curve ViSoBERT (mốc {sizes}, {seeds} seed, {epochs} epoch, {tiers} tầng): '
      f'~{est / 60:.0f} phút GPU, chưa tính thời gian đánh giá trên Validation.')

## Learning curve của ViSoBERT

Sau khi chắc môi trường chạy được, em bắt đầu phần chính của giai đoạn này: xem hiệu năng thay đổi thế nào khi tăng số review gán
nhãn, để biết 1.200 review Train đã đủ hay cần gán thêm. Theo protocol, em dùng ViSoBERT làm mô hình chính cho learning curve, với
ba mốc 400, 800 và 1.200 review, mỗi mốc ba seed (42, 1337, 2024), huấn luyện riêng tầng 1 (ACTIONABLE / NOT_ACTIONABLE) và tầng 2
(BUG / FEATURE / UX), và luôn đánh giá trên cùng một tập Validation cố định.

Có hai chi tiết em làm theo đề cương để đường cong ít nhiễu. Thứ nhất, các tập con được lồng nhau: tập 400 nằm trong tập 800, tập
800 nằm trong tập 1.200, và mỗi tập con giữ đúng tỷ lệ các lớp như Train, để lớp hiếm như FEATURE không bị mất ở mốc nhỏ. Thứ hai,
tập con cố định theo một seed riêng, chỉ seed huấn luyện thay đổi, nên chênh lệch giữa các mốc phản ánh lượng dữ liệu chứ không phải
việc rút mẫu khác nhau. Ở tầng 2, em bỏ các review được đánh dấu ambiguous khỏi dữ liệu huấn luyện như protocol quy định, nhưng vẫn
giữ chúng khi đánh giá trên Validation. Ở bước này em dùng ngưỡng mặc định 0,5 cho tầng 1; việc chọn ngưỡng theo yêu cầu recall
của Q5 (amendment v2.0: ≥ 0,80) thuộc bước RQ1 sau khi đã chốt kích thước Train.

Toàn bộ phần huấn luyện và đánh giá nằm trong `src/classification/`. Mỗi lần chạy xong được ghi ngay vào file CSV, nên nếu phiên
Kaggle bị ngắt giữa chừng em có thể chạy lại cell và nó sẽ tiếp tục từ chỗ dừng.

In [ ]:
import gc
from src.utils.kaggle_paths import working_dir
from src.classification.learning_curve import (run_learning_curve, summarize, stopping_evidence,
                                               plot_learning_curve, nested_subsets)
from src.classification.strategy_visobert.train import train_visobert
from src.classification.strategy_visobert.predict import predict_visobert

for _name in ('model', 'opt'):              # giải phóng mô hình của lần chạy thử
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

VCFG = CFG['strategy_b_visobert']
SIZES, SEEDS = CFG['learning_curve_sizes'], CFG['learning_curve_seeds']
OUT = working_dir('learning_curve')
RAW_CSV = os.path.join(OUT, 'learning_curve_raw.csv')
print('Cấu hình:', VCFG, '| mốc:', SIZES, '| seed:', SEEDS, '| lưu tại:', OUT)
for n, d in nested_subsets(train_df, SIZES).items():
    print(n, d['gold_label'].value_counts().reindex(order).to_dict())

In [ ]:
_tok = {}
def visobert_train_fn(df, n_classes, label_names, seed):
    model, tok, hist = train_visobert(df, n_classes, label_names, VCFG, seed)
    _tok['tok'] = tok
    return model

def visobert_predict_fn(model, texts):
    return predict_visobert(model, _tok['tok'], texts, VCFG['max_len'])[0]

lc_raw = run_learning_curve(train_df, val_df, SIZES, SEEDS, visobert_train_fn, visobert_predict_fn,
                            strategy='visobert', checkpoint_csv=RAW_CSV)
print('Số dòng kết quả:', len(lc_raw), '| tổng thời gian huấn luyện:',
      round(lc_raw.drop_duplicates(['size', 'tier', 'seed'])['train_seconds'].sum() / 60, 1), 'phút')

In [ ]:
lc_sum = summarize(lc_raw)
lc_sum.to_csv(os.path.join(OUT, 'learning_curve_summary.csv'), index=False)
show = lc_sum.assign(value=lambda d: (d['mean'] * 100).round(1).astype(str) + ' ± ' + (d['std'] * 100).round(1).astype(str))
for tier, metrics in [('tier1', ['f1_actionable', 'recall_actionable', 'precision_actionable', 'macro_f1']),
                      ('tier2', ['macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX'])]:
    t = show[(show['strategy'] == 'visobert') & (show['tier'] == tier)].pivot(index='size', columns='metric', values='value')
    print(f'{tier} (F1 %, trung bình ± độ lệch chuẩn qua {len(SEEDS)} seed):')
    print(t[metrics].to_string(), '\n')

In [ ]:
fig_path = plot_learning_curve(lc_sum, os.path.join(OUT, 'learning_curve_visobert.png'), strategy='visobert')
from IPython.display import Image, display
display(Image(fig_path))

### Bằng chứng về điểm bão hòa

Protocol đặt quy tắc dừng gán nhãn trước khi có kết quả: coi là bão hòa khi hiệu năng trên Validation tăng chưa tới khoảng 1 điểm
phần trăm ở hai lần tăng dữ liệu liên tiếp, dải dao động giữa các seed chồng lấp nhau, và F1 của từng lớp BUG / FEATURE / UX không
còn tăng thực chất; nếu điểm tổng gần phẳng mà FEATURE vẫn tăng rõ thì vẫn nên gán thêm. Với ba mốc hiện có, em có đúng hai lần tăng
dữ liệu (400 → 800 và 800 → 1.200). Cell dưới đây chỉ in các con số để đối chiếu với quy tắc đó; em chưa kết luận dừng hay gán thêm
ở đây.

In [ ]:
for line in stopping_evidence(lc_sum, strategy='visobert'):
    print(line)

## Kiểm tra nhanh với Bi-LSTM + fastText

Theo protocol, Bi-LSTM chỉ cần kiểm tra ở hai mốc để xem xu hướng có khác ViSoBERT hay không, nên em chạy mốc 400 và 1.200 với
một seed. Mô hình dùng vector fastText tiếng Việt đã huấn luyện sẵn trên Common Crawl và Wikipedia (`cc.vi.300`), giữ cố định
không huấn luyện lại, đúng như đề cương. Vì từ điển của bộ vector này là các âm tiết tách bằng khoảng trắng, em tách văn bản theo
âm tiết thay vì nối từ ghép bằng dấu gạch dưới, để phần lớn token có vector. File vector khá lớn (khoảng 1 GB) nên có thể tải không
được; khi đó em ghi nhận là chưa chạy Bi-LSTM và để sang Giai đoạn 6, phần ViSoBERT ở trên không bị ảnh hưởng.

In [ ]:
from src.classification.strategy_bilstm_fasttext.embeddings import build_vocab, download, load_vectors
from src.classification.strategy_bilstm_fasttext.train import train_bilstm, predict_bilstm
BCFG = CFG['strategy_a_bilstm']
try:
    vec_path = download(BCFG['fasttext_url'], '/tmp/cc.vi.300.vec.gz')
    vocab = build_vocab(pd.concat([train_df['text'], val_df['text']]).astype(str))
    emb, coverage = load_vectors(vec_path, vocab, dim=int(BCFG['embedding_dim']))
    print(f'Từ vựng: {len(vocab):,} | có vector fastText: {coverage:.1%}')
    bl_train = lambda df, k, names, seed: train_bilstm(df, k, names, BCFG, seed, vocab, emb)[0]
    bl_pred = lambda model, texts: predict_bilstm(model, texts, vocab, int(BCFG['max_len']))[0]
    bl_raw = run_learning_curve(train_df, val_df, [400, 1200], [42], bl_train, bl_pred, strategy='bilstm',
                                checkpoint_csv=os.path.join(OUT, 'learning_curve_bilstm_raw.csv'))
    bl_sum = summarize(bl_raw)
    bl_sum.to_csv(os.path.join(OUT, 'learning_curve_bilstm_summary.csv'), index=False)
    for line in stopping_evidence(bl_sum, strategy='bilstm'):
        print(line)
except Exception as e:
    print('Chưa chạy Bi-LSTM, để sang GĐ6. Lý do:', repr(e))

## Nhận xét

Các con số dưới đây là của lần chạy lại sau khi em sửa lỗi tokenizer của ViSoBERT. Ở lần chạy đầu, `AutoTokenizer` tách từ sai (xem
phần GPU và thư viện ở trên), làm F1 ACTIONABLE ở mốc 1.200 thấp hơn 0,138 và Macro-F1 tầng 2 thấp hơn 0,080 so với khi tách đúng.
Kết quả của lần chạy lỗi em vẫn giữ trong `05_experiments/_tokenizer_bug/` để đối chiếu, nhưng không dùng cho phân tích.

Ở tầng 1, F1 của lớp ACTIONABLE là 0,706 ± 0,035 ở 400 review, 0,708 ± 0,028 ở 800 và 0,735 ± 0,018 ở 1.200. Từ 400 lên 800 gần như
không đổi, từ 800 lên 1.200 tăng 2,7 điểm, nhiều hơn ngưỡng dừng khoảng 1 điểm nhưng chỉ cỡ độ lệch chuẩn giữa ba seed. Vì vậy em
chỉ nói được là tầng 1 tăng chậm, chưa khẳng định được là đã bão hòa. Recall ACTIONABLE ở cấu hình mặc định (4 epoch, lr 2e-5,
ngưỡng 0,5) là 0,648 / 0,648 / 0,671, còn xa mục tiêu recall của protocol; phần này phụ thuộc nhiều vào ngưỡng, nên việc tune và
chọn ngưỡng em làm ở bước RQ1.

Tầng 2 tăng rõ và đều qua ba mốc: Macro-F1 0,578 → 0,670 → 0,735. Từ 800 lên 1.200 review, F1 của BUG tăng 5,8 điểm (0,685 → 0,743),
F1 của FEATURE tăng 8,3 điểm (0,556 → 0,639) và Macro-F1 tăng 6,5 điểm, đều vượt xa ngưỡng dừng khoảng 1 điểm. Theo quy tắc dừng,
tầng 2 chưa bão hòa và về nguyên tắc nên gán thêm Train. Validation chỉ có 22–27 review cho mỗi lớp actionable, nên F1 từng lớp vẫn
dao động khá mạnh, nhất là FEATURE ở mốc 400 (độ lệch chuẩn 11,2 điểm), và em đọc từng con số một cách thận trọng.

Lớp còn tăng mạnh là FEATURE, lớp hiếm nhất (khoảng 2,5% trong dữ liệu tự nhiên), và khi em kiểm tra pool Shopee còn lại thì không
còn review nào khớp các cụm từ gợi ý FEATURE: Train và focused set đã dùng hết ở Giai đoạn 3. Gán thêm ngẫu nhiên chỉ được thêm
khoảng 2,5% FEATURE với chi phí gán đôi rất lớn, còn làm giàu BUG thì không giải quyết đúng chỗ nghẽn. Vì vậy chúng em quyết định dừng
Train ở 1.200 review (amendment v1.9) và coi việc tầng 2 chưa bão hòa là giới hạn của dữ liệu thực chứ không phải của phương pháp. Để
bù cho điều này, F1 của FEATURE ở tầng 2 sẽ được báo cáo thêm trên focused FEATURE/UX set (56 review FEATURE), bên cạnh Hold-out. Vì
cả ba chiến lược đều dùng chung dữ liệu Train 1.200 này, việc so sánh giữa chúng ở RQ1 vẫn công bằng.

Phần kiểm tra nhanh Bi-LSTM + fastText (một seed, không dùng tokenizer của ViSoBERT nên không bị lỗi trên ảnh hưởng) cho F1 ACTIONABLE
tăng từ 0,567 lên 0,662 giữa 400 và 1.200 review, còn Macro-F1 tầng 2 gần như giữ nguyên (0,689 → 0,700). Ở mốc 1.200, Bi-LSTM thấp
hơn ViSoBERT ở cả hai tầng (0,662 so với 0,735 ở tầng 1, 0,700 so với 0,735 ở tầng 2). Toàn bộ 18 lần huấn luyện ViSoBERT của
learning curve chỉ mất khoảng 4,3 phút GPU.

Các file kết quả (`learning_curve_raw.csv`, `learning_curve_summary.csv`, `learning_curve_visobert.png` và phần Bi-LSTM) đã được em tải
về `05_experiments/learning_curve/`.